# Lab 1: SELF-ASSESSMENT: Profiling LLM Inference Across Hardware

**Course**: *Dataflow Architectures for AI: From Principles to Practice*  
**Week**: 1, The Accelerator Landscape  
**Instructor**: Dr. Kaoutar El Maghraoui (IBM Research)  
**Lab Instructor**: Dr. Rashed Z. Bhatti (IBM Research)  
**Runtime**: Coursera Labs (CPU only — **no GPU needed here**)  
**Expected time**: 15–20 minutes

---

> **Self-Assessment notebook — the 40 points here are ungraded.** This is where
> you check how well your Lab 1 run went. You do not write code or run
> measurements here — you paste the `lab1_results.json` your Colab run produced,
> and all **40 ungraded Self-Assessment points** are checked from it
> automatically. There is no written analysis and no human review.
>
> **The graded item for this lab is the separate Concept Check quiz** (10 points,
> auto-graded in the Coursera Graded Assignment section). It is mandatory and
> requires **80%** to pass. Nothing in *this* notebook affects your course grade.

## Learning objectives
This notebook assesses the objectives of the Colab lab:
1. Build a repeatable benchmarking harness with torch.utils.benchmark for CPU and GPU inference.
2. Profile memory allocation during inference with torch.profiler and the CUDA memory stats.
3. Construct the first row of the Accelerator Landscape Map from measurements you made yourself.
4. Explain the CPU vs GPU gap in terms of SIMT parallelism, device-memory bandwidth, and precision.

## Prerequisites
A completed run of
[Lab1_Profiling_Inference_STUDENT.ipynb](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/gpu-labs/student/Lab1_Profiling_Inference_STUDENT.ipynb)
on Google Colab (free T4), with its final cell run so that **`lab1_results.json`**
exists.

## Why this notebook exists

Lab 1 measures LLM inference on a real GPU, so you run it on **Google Colab**
(free T4). This Coursera notebook is where your results get **checked, for your
own feedback**.

The grading environment has **no GPU, no model download, and no network
access**, so it cannot re-run your measurements. What it *can* do — and what it
does — is check that the numbers/values you measured are **internally consistent**:
throughput has to follow from latency, speedup from the two throughputs, the
memory ratio from the byte counts, the parameter count from both. Those
relationships hold for any real run on any GPU, and they break when values are
edited by hand.

## The workflow

| Step | Where | What you do |
|------|-------|-------------|
| 1 | GitHub → Colab | Open [Lab1_Profiling_Inference_STUDENT.ipynb](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/gpu-labs/student/Lab1_Profiling_Inference_STUDENT.ipynb), then **File → Save a copy in Drive** |
| 2 | Colab | Set **Runtime → Change runtime type → T4 GPU**, then complete every TODO and run all cells |
| 3 | Colab | Run the final cell. It writes **`lab1_results.json`** |
| 4 | Colab | Copy the JSON it prints, or open the file browser (📁, left sidebar) and double-click `lab1_results.json` |
| 5 | **Here** | Paste it into the `RESULTS_JSON` cell at the top of this notebook, then **Kernel → Restart & Run All** for your self-assessment |

> **The free Colab T4 is all you need.** Every point below is reachable with a
> CPU measurement and a T4 measurement — exactly what the free tier gives you.
> You do not need Colab Pro, and you do not need an L4, V100 or A100. If Colab
> happens to assign you a different GPU, your self-assessment is unaffected.
>
> **Students are encouraged to experiment with other GPUs and accelerators to
> expand their learning.** Comparing a T4 against an L4 or A100 is genuinely
> instructive, and Part C checks the ridge point against whichever GPU you
> actually ran on. There is **no extra credit** for doing so; all 40
> Self-Assessment points come from a single free-tier run.

## How this notebook is checked (40 ungraded Self-Assessment points)

| Part | Points | What it checks |
|------|-------:|----------------|
| **A — Structural invariants** | 14 | Facts true on *any* GPU: arithmetic intensity, model identity, required fields, units |
| **B — Self-consistency** | 18 | Your derived values recomputed from your own raw measurements |
| **C — Physics & plausibility** | 8 | Ordering, wide sanity bands, and the roofline verdict |
| **Ungraded notebook total** | **40** | fully auto-graded from your JSON for Self-Assessment |

**40 points is a full self-assessment score, and a CPU + free-tier T4 run earns
all of it.** Every check is **auto-graded from the JSON you paste** — there is no
written analysis and no human review. Parts A and B (32 points) are
hardware-independent; Part C uses wide bands and checks the ridge point against
whichever GPU you actually ran on. A T4, L4, V100 or A100 run all score the same.

A **score table** near the end of this notebook reports your 40-point
self-assessment total, so you know exactly which parts of the lab went well.

**Graded Quiz points:**  
A 10 point graded Coursera Assignment quiz is given to assess your understanding
from this lab experience. It is **mandatory**, and you need **80%** to pass.

| Quiz | Points | Comes from |
|---|---:|---|
| Concept Check quiz | 10 | Auto-graded Coursera Assignment |

> **Rounding is fine.** Every numeric check allows 2% relative tolerance, which
> is far more than the rounding in the lab's printed output. Transfer the values
> as the lab produced them and you will be well inside it.

> **Do not skip the Concept Check quiz.** It lives in the Coursera **Graded
> Assignment** section, is auto-graded as its own item, and is worth **10 graded
> points** — the only points for this lab that count toward your course grade.
> It is mandatory and requires 80% to pass.


## Use your own measurements

These 40 points are ungraded, so there is nothing to gain by editing numbers —
the value of this notebook is the feedback. Use measurements from **your own**
Colab run of Lab 1.

Be aware that the checks in Part B are *cross-checks*: they recompute each
derived quantity from your own raw inputs and require the two to agree. Numbers
that were not produced together by a real run generally fail several of these
at once. Transcription and rounding are fine; invented values are not.

---
## Your results  &nbsp;·&nbsp; *start here*

Paste the **entire contents** of `lab1_results.json` from your Colab run into the
next cell, between the triple quotes — from the opening `{` through the closing `}`.

Keep the raw-string wrapper (the `r` and the triple quotes) exactly as it is: it
means backslashes and quotes inside your JSON cannot break the cell.

```python 
RESULTS_JSON = r"""
paste the contents of lab1_results.json here
"""
```

> **This cell only stores your text — it does not check it yet.** After pasting,
> use **Kernel → Restart & Run All**: the Setup cell below loads the helpers, and
> every check in Parts A–C runs against what you pasted here.

> **If you cannot paste**, upload `lab1_results.json` next to this notebook
> instead (Coursera Labs: the **upload** button in the file browser) and leave the
> placeholder text alone. The loader checks the pasted text first, then falls back
> to the file.



In [ ]:
# Paste the contents of lab1_results.json between the triple quotes.
# (Leave the surrounding raw-string triple quotes exactly as they are.)
# This cell only stores the text; the checks run after Kernel -> Restart & Run All.

# your code here
raise NotImplementedError
RESULTS_JSON = r"""
paste the contents of lab1_results.json here
"""


---
## Setup

The next two cells are **read-only — just run them.** The first defines the
loader and the comparison helpers used by every check below; the second loads
the JSON you pasted above and echoes it back so you can confirm it arrived
intact. They import only the Python standard library: nothing here needs
`torch`, a GPU, or the network.



In [ ]:
# === Grading helpers (read-only) =======================================
# Standard library only: nothing here needs torch, a GPU, or the network.
import json, ast, os, re

TOL  = 0.02          # 2% relative tolerance on derived quantities
B, S = 1, 64         # BATCH_SIZE and SEQ_LEN used throughout Lab 1

# The five keys the original lab exported, and the six added so this notebook
# can CHECK your Part 3 work instead of making you retype the formulas.
CORE_KEYS = ("perf", "peak_bytes", "mem_overhead_ratio", "top3", "landscape_row")
PART3_KEYS = ("weights_bytes", "n_params", "ai", "gpu_name", "ridge_point", "verdict")

REQUIRED_LS_KEYS = {
    "accelerator", "execution_model", "memory_hierarchy", "programming_model",
    "forward_latency_ms", "throughput_tps", "peak_memory_MB", "dominant_op",
}

# Ridge point = peak fp16 TFLOPS / peak memory bandwidth (TB/s). It genuinely
# differs per GPU, so it can only be range-checked -- never equality-checked
# against the T4 value. An unlisted GPU falls through to the wide band.
RIDGE_TABLE = {
    "t4":   65.0 / 0.320,     # 203.1
    "l4":   121.0 / 0.300,    # 403.3
    "v100": 125.0 / 0.900,    # 138.9
    "a100": 312.0 / 2.039,    # 153.0
}
RIDGE_BAND = (100.0, 450.0)
GPU_FAMILIES = ("t4", "l4", "v100", "a100", "p100", "h100", "rtx", "titan")

# Matrix-multiply kernels are named differently on each GPU architecture
# (Turing: turing_fp16_s1688gemm_*, Ampere: ampere_*/sm80_xmma_*), so we match
# the GEMM *family* instead of any one vendor kernel name.
GEMM_TOKENS = ("mm", "gemm", "matmul", "linear", "addmm", "bmm",
               "xmma", "cutlass", "sgemm")

# --- score bookkeeping (for the courtesy readout near the end) ----------
# Each graded cell calls record(...) as its LAST line, after its asserts. A
# cell that raises never reaches that line, so "absent" means "failed".
_SCORE = {}

def record(gid):
    """Mark a graded cell as passed. Called at the end of each graded cell."""
    _SCORE[gid] = True

# Every autograded cell, with its part and points. The tally walks this, so it
# must stay in step with the cells' nbgrader metadata (a test enforces that).
MANIFEST = [
    ("a_units_peak_bytes",        "A", 2),
    ("a_model_identity",          "A", 3),
    ("a_landscape_keys",          "A", 2),
    ("a_landscape_types",         "A", 1),
    ("a_top3_shape",              "A", 2),
    ("a_no_wrapper",              "A", 2),
    ("a_ai_equals_64",            "A", 2),
    ("b_cpu_tps",                 "B", 2),
    ("b_gpu_tps",                 "B", 2),
    ("b_speedup_vs_tps",          "B", 2),
    ("b_speedup_vs_ms",           "B", 2),
    ("b_mem_ratio",               "B", 3),
    ("b_ls_latency",              "B", 2),
    ("b_ls_tps",                  "B", 1),
    ("b_ls_peak_mb",              "B", 2),
    ("b_ls_dominant_op",          "B", 2),
    ("c_cpu_gt_gpu",              "C", 1),
    ("c_cpu_ms_plausible",        "C", 1),
    ("c_gpu_ms_plausible",        "C", 1),
    ("c_speedup_gt",              "C", 2),
    ("c_ratio_ge_1",              "C", 1),
    ("c_memory_bound",            "C", 2),
]
CHECK_LABELS = {
    "a_units_peak_bytes":    "peak_bytes is raw bytes, not megabytes",
    "a_model_identity":      "parameter count agrees via two routes",
    "a_landscape_keys":      "landscape_row has all 8 fields, filled",
    "a_landscape_types":     "landscape_row field types (numeric vs text)",
    "a_top3_shape":          "top3 is three (operator, time_us) pairs",
    "a_no_wrapper":          "llm_forward wrapper filtered out of top3",
    "a_ai_equals_64":        "arithmetic intensity == batch x seq_len",
    "b_cpu_tps":             "cpu_tps follows from cpu_ms",
    "b_gpu_tps":             "gpu_tps follows from gpu_ms",
    "b_speedup_vs_tps":      "speedup == throughput ratio",
    "b_speedup_vs_ms":       "speedup == latency ratio",
    "b_mem_ratio":           "mem_overhead_ratio == peak / weights",
    "b_ls_latency":          "row latency mirrors perf['gpu_ms']",
    "b_ls_tps":              "row throughput mirrors perf['gpu_tps']",
    "b_ls_peak_mb":          "peak_memory_MB unit conversion",
    "b_ls_dominant_op":      "dominant_op is top entry of top3",
    "c_cpu_gt_gpu":          "CPU forward slower than GPU",
    "c_cpu_ms_plausible":    "CPU latency in plausible band",
    "c_gpu_ms_plausible":    "GPU latency in plausible band",
    "c_speedup_gt":          "run really used a GPU runtime",
    "c_ratio_ge_1":          "peak memory exceeds weights alone",
    "c_memory_bound":        "roofline verdict",
}

PART_NAMES = {
    "A": "Structural invariants",
    "B": "Self-consistency",
    "C": "Physics and plausibility",
}


def is_gemm(name):
    """True if an operator name looks like a matrix-multiply kernel."""
    return any(t in str(name).lower() for t in GEMM_TOKENS)


def rel(a, b):
    """Relative difference between a and b, safe at b == 0."""
    return abs(float(a) - float(b)) / max(abs(float(b)), 1e-12)


def gpu_family(text):
    """The set of known GPU model tokens appearing in a device string."""
    t = str(text).lower()
    return set(f for f in GPU_FAMILIES if f in t)


def num(x, field="value"):
    """Read x as a float, forgiving the ways a real export can mangle one.

    The lab writes plain JSON, but students paste through editors and older
    notebook versions used default=str, so a value can arrive as "1271.0",
    "1030.5 MB", "tensor(64.)" or "np.float64(64.0)" instead of a number.
    Coerce rather than fail: a unit suffix or a repr wrapper is a formatting
    accident, not a wrong answer.
    """
    if isinstance(x, bool):
        raise AssertionError(f"{field}: expected a number, got a boolean.")
    if x is None:
        raise AssertionError(
            f"{field}: is missing (None). Re-run the final cell of the Colab lab.")
    if isinstance(x, (int, float)):
        return float(x)
    if isinstance(x, str):
        t = x.strip().replace(",", "")
        # tensor(64.) / np.float64(64.0) / Decimal('64') -> 64
        m = re.fullmatch(r"[A-Za-z_][A-Za-z0-9_.]*\(\s*['\"]?([-\d.eE+]+)['\"]?\s*\)", t)
        if m:
            t = m.group(1)
        for suffix in ("bytes", "tok/s", "tokens/s", "MB", "mb", "GB", "ms", "us", "x"):
            if t.lower().endswith(suffix.lower()):
                t = t[: -len(suffix)].strip()
        try:
            return float(t)
        except ValueError:
            pass
    raise AssertionError(f"{field}: could not read {x!r} as a number.")


def _parse(text):
    text = text.strip()
    if not text:
        raise AssertionError("No results provided.")
    try:
        return json.loads(text)                 # normal path
    except Exception:
        pass
    try:
        return ast.literal_eval(text)           # tolerate a pasted Python dict
    except Exception:
        pass
    if text.startswith("{") and not text.endswith("}"):
        raise AssertionError(
            "Your pasted JSON looks incomplete. Copy from the opening '{' "
            "through the final '}'.")
    raise AssertionError(
        "Could not parse your results. Paste the exact contents of "
        "lab1_results.json. (Curly 'smart quotes' from a word processor "
        "will break JSON - paste as plain text.)")


def load_results(blob=""):
    """Pasted JSON first, then an uploaded lab1_results.json, else explain."""
    data, src = None, None
    if blob and blob.strip() and "paste the contents" not in blob:
        data, src = _parse(blob), "pasted text"
    else:
        for p in ("lab1_results.json", "submission/lab1_results.json",
                  os.path.expanduser("~/lab1_results.json")):
            if os.path.exists(p):
                with open(p) as f:
                    data, src = _parse(f.read()), p
                break
    if data is None:
        raise AssertionError(
            "No results found. Either paste the contents of lab1_results.json "
            "into RESULTS_JSON near the top of this notebook, or upload the "
            "file next to this notebook.")
    if not isinstance(data, dict):
        raise AssertionError("Your results should be a JSON object (a dict).")

    # Two tiers of missing-key help. A missing CORE key means the lab did not
    # finish; a missing PART3 key means the notebook is an older version.
    missing_core = [k for k in CORE_KEYS if k not in data]
    if missing_core:
        raise AssertionError(
            f"Key(s) {missing_core} are missing from your results. Re-run the "
            f"final cell of the Colab lab to regenerate lab1_results.json.")
    missing_p3 = [k for k in PART3_KEYS if k not in data]
    if missing_p3:
        raise AssertionError(
            f"Key(s) {missing_p3} are missing. Your Colab notebook is an older "
            f"version of Lab 1: re-download Lab1_Profiling_Inference_STUDENT."
            f"ipynb, re-run it, and copy the new lab1_results.json. (The "
            f"current Part 7 cell exports 11 top-level keys.)")

    perf_in = {str(k).strip().lower(): v for k, v in dict(data["perf"]).items()}
    need = ("cpu_ms", "gpu_ms", "cpu_tps", "gpu_tps", "speedup")
    missing = [k for k in need if k not in perf_in]
    if missing:
        raise AssertionError(f"perf is missing {missing}. Did every Part 2 cell run?")
    perf = {k: num(perf_in[k], f"perf['{k}']") for k in need}

    peak_bytes    = num(data["peak_bytes"], "peak_bytes")
    weights_bytes = num(data["weights_bytes"], "weights_bytes")
    n_params      = num(data["n_params"], "n_params")
    ratio         = num(data["mem_overhead_ratio"], "mem_overhead_ratio")
    ai            = num(data["ai"], "ai")
    ridge_point   = num(data["ridge_point"], "ridge_point")

    if ratio <= 0:
        raise AssertionError("mem_overhead_ratio must be positive.")
    if weights_bytes <= 0:
        raise AssertionError("weights_bytes must be positive.")

    verdict = str(data["verdict"]).strip().upper().replace(" ", "-").replace("_", "-")
    if verdict not in ("MEMORY-BOUND", "COMPUTE-BOUND"):
        raise AssertionError(
            f"verdict should be 'MEMORY-BOUND' or 'COMPUTE-BOUND', got "
            f"{data['verdict']!r}.")

    # JSON turns the (name, time) tuples into 2-element lists. Accept either.
    top3 = []
    for e in data["top3"]:
        if not isinstance(e, (list, tuple)) or len(e) != 2:
            raise AssertionError(
                "Each top3 entry should be a (operator_name, time_us) pair.")
        top3.append((str(e[0]), num(e[1], "a top3 time")))

    # Keep BOTH paths to the parameter count. The exported value says what your
    # lab measured directly; the derived value reconstructs it from the two byte
    # figures. Part A grades that they agree -- which is the check a hand-edited
    # results file fails.
    n_params_derived = peak_bytes / ratio / 2.0

    return {
        "perf": perf,
        "peak_bytes": peak_bytes,
        "weights_bytes": weights_bytes,
        "n_params": n_params,
        "n_params_derived": n_params_derived,
        "mem_overhead_ratio": ratio,
        "ai": ai,
        "gpu_name": str(data["gpu_name"]),
        "ridge_point": ridge_point,
        "verdict": verdict,
        "top3": top3,
        "landscape_row": dict(data["landscape_row"]),
        "_source": src,
    }


print("Helpers loaded. Tolerance on derived values: "
      f"{TOL*100:.0f}%   (batch={B}, seq_len={S})")
print(f"Self-Assessment checks: {len(MANIFEST)}   "
      f"Parts A-C total: {sum(p for _, _part, p in MANIFEST)} ungraded points"
      f"   (fully auto-graded, no human review)")


In [ ]:
# === Load and echo your results (read-only) ============================
# Every graded cell below calls load_results() itself, so one bad field can
# never cascade into unrelated failures.
_SCORE.clear()   # fresh top-to-bottom run: forget any earlier cell outcomes

_r = load_results(RESULTS_JSON)
_p = _r["perf"]

print(f"Loaded from: {_r['_source']}\n")
print(f"  CPU     {_p['cpu_ms']:10.1f} ms/fwd   {_p['cpu_tps']:9.1f} tok/s")
print(f"  GPU     {_p['gpu_ms']:10.1f} ms/fwd   {_p['gpu_tps']:9.1f} tok/s")
print(f"  Speedup {_p['speedup']:10.1f}x\n")
print(f"  peak_bytes          {_r['peak_bytes']:>15,.0f} bytes "
      f"({_r['peak_bytes']/1e6:.1f} MB)")
print(f"  weights_bytes       {_r['weights_bytes']:>15,.0f} bytes "
      f"({_r['weights_bytes']/1e6:.1f} MB)")
print(f"  mem_overhead_ratio  {_r['mem_overhead_ratio']:>15.3f}x")
print(f"  n_params (reported) {_r['n_params']/1e6:>15.1f} M")
print(f"  n_params (derived)  {_r['n_params_derived']/1e6:>15.1f} M "
      f"<- from peak_bytes / ratio / 2; Part A checks these agree\n")
print(f"  arithmetic intensity {_r['ai']:>14.1f} FLOPs/byte")
print(f"  GPU                  {_r['gpu_name']:>14s}")
print(f"  ridge point          {_r['ridge_point']:>14.1f} FLOPs/byte")
print(f"  verdict              {_r['verdict']:>14s}\n")
print("  Top-3 operators by self-CUDA time:")
for _n, _us in _r["top3"]:
    print(f"    {_n[:52]:52s} {_us/1000:8.2f} ms")
print(f"\n  Accelerator: {_r['landscape_row'].get('accelerator')}")
print(f"  Dominant op: {_r['landscape_row'].get('dominant_op')}")
print("\nCheck these against your Colab output before continuing.")


---
# Part A — Structural invariants (14 ungraded points)

These hold for **any** correct run on **any** GPU. They check that your results
are complete, in the right units, and describe the model the lab actually used.

In [ ]:
# A1 (2 pts): peak_bytes must be raw BYTES from torch.cuda.max_memory_allocated().
_r = load_results(RESULTS_JSON)
_pb = _r["peak_bytes"]
assert _pb > 1e5, (
    f"peak_bytes = {_pb:g} looks like megabytes, not bytes. Use the raw value "
    f"from torch.cuda.max_memory_allocated() (expect roughly 1e9 for this lab).")
assert _pb < 1e12, f"peak_bytes = {_pb:g} is implausibly large for a 0.5B model."
print(f"[OK] peak_bytes = {_pb:,.0f} bytes ({_pb/1e6:.1f} MB)")
record("a_units_peak_bytes")


In [ ]:
# A2 (3 pts): your two independent routes to the parameter count must agree.
#
#   exported : n_params, measured directly in the lab
#   derived  : peak_bytes / mem_overhead_ratio / 2   (fp16 = 2 bytes/param)
#
# Both describe the same model, so they have to land on the same number -- and
# on Qwen2.5-0.5B-Instruct's ~494 M. This is the check that a hand-edited
# results file fails: moving any one of peak_bytes, mem_overhead_ratio,
# weights_bytes or n_params breaks the agreement.
_r = load_results(RESULTS_JSON)
_n, _nd = _r["n_params"], _r["n_params_derived"]

_e = rel(_n, _nd)
assert _e <= TOL, (
    f"Your exported n_params = {_n/1e6:.1f} M but peak_bytes / "
    f"mem_overhead_ratio / 2 = {_nd/1e6:.1f} M, off by {_e*100:.1f}%. These "
    f"must agree. Check that mem_overhead_ratio = peak_bytes / weights_bytes "
    f"and that both byte counts are in BYTES.")

assert 4.0e8 <= _n <= 6.0e8, (
    f"Your numbers imply {_n/1e6:.1f} M parameters, but "
    f"Qwen/Qwen2.5-0.5B-Instruct has about 494 M.")

# fp16 weights: exactly 2 bytes per parameter.
_e2 = rel(_r["weights_bytes"], 2 * _n)
assert _e2 <= TOL, (
    f"weights_bytes = {_r['weights_bytes']:,.0f} but 2 x n_params = "
    f"{2*_n:,.0f}. The GPU model is fp16, so weights are 2 bytes per "
    f"parameter (off by {_e2*100:.1f}%).")

print(f"[OK] parameter count agrees both ways: exported {_n/1e6:.1f} M, "
      f"derived {_nd/1e6:.1f} M (expected about 494 M)")
record("a_model_identity")


In [ ]:
# A3 (2 pts): landscape_row must have exactly the 8 required fields, all filled.
_r = load_results(RESULTS_JSON)
_ls = _r["landscape_row"]
_got = set(_ls)
assert _got == REQUIRED_LS_KEYS, (
    f"landscape_row keys are wrong.\n  missing: {sorted(REQUIRED_LS_KEYS - _got)}"
    f"\n  unexpected: {sorted(_got - REQUIRED_LS_KEYS)}")
for _k, _v in _ls.items():
    assert _v is not None, f"landscape_row['{_k}'] is still None."
print("[OK] all 8 landscape_row fields present and filled")
record("a_landscape_keys")


In [ ]:
# A4 (1 pt): the three measured fields are numeric, the four descriptors are text.
_r = load_results(RESULTS_JSON)
_ls = _r["landscape_row"]
for _k in ("forward_latency_ms", "throughput_tps", "peak_memory_MB"):
    num(_ls[_k], f"landscape_row['{_k}']")
for _k in ("accelerator", "execution_model", "memory_hierarchy", "programming_model"):
    assert isinstance(_ls[_k], str) and _ls[_k].strip(), (
        f"landscape_row['{_k}'] should be a short non-empty description.")
print("[OK] landscape_row field types correct")
record("a_landscape_types")


In [ ]:
# A5 (2 pts): top3 holds exactly three (operator_name, time_us) pairs.
_r = load_results(RESULTS_JSON)
_t3 = _r["top3"]
assert len(_t3) == 3, f"top3 must have exactly 3 entries, found {len(_t3)}."
for _name, _us in _t3:
    assert _name.strip(), "An operator name in top3 is empty."
    assert _us >= 0, f"Operator '{_name}' has a negative time."
print("[OK] top3 shape correct: " + ", ".join(n for n, _ in _t3))
record("a_top3_shape")


In [ ]:
# A6 (2 pts): the record_function wrapper must be filtered out of top3.
_r = load_results(RESULTS_JSON)
for _name, _ in _r["top3"]:
    assert _name.strip().lower() != "llm_forward", (
        "'llm_forward' is the record_function label wrapping the whole forward "
        "pass, not a real operator. Filter it out before taking the top 3, or "
        "it hides the true dominant operator.")
print("[OK] 'llm_forward' correctly excluded from top3")
record("a_no_wrapper")


---
## Arithmetic intensity and the roofline verdict

Lab 1 computes both of these in **Part 3** and exports them, so there is
nothing to retype here:

- `ai = approx_flops / weight_bytes`, which works out to `batch x seq_len` on
  any GPU and for any model size, because the parameter count cancels:
  `(2*N*B*S) / (2*N) = B*S`.
- `ridge_point = peak fp16 TFLOPS / peak memory bandwidth (TB/s)`, which is
  specific to the GPU you ran on.
- `verdict`, which is `"MEMORY-BOUND"` when `ai < ridge_point`.

The echo table above shows all three. **Part A** checks the arithmetic
intensity, and **Part C** checks the verdict — including that the ridge point
you reported belongs to the GPU your run actually used.

| GPU | FP16 TFLOPS | BW (TB/s) | Ridge point (FLOPs/byte) | |
|-----|------------:|----------:|-------------------------:|---|
| **T4** |  **65.0** | **0.320** | **203** | **Colab free tier** |
| L4   | 121.0 | 0.300 | 403 | only if Colab assigned it |
| V100 | 125.0 | 0.900 | 139 | only if Colab assigned it |
| A100 | 312.0 | 2.039 | 153 | only if Colab assigned it |


In [ ]:
# A7 (2 pts): arithmetic intensity is exactly batch x seq_len, on every GPU.
#
#   ai = FLOPs / bytes = (2 * N * B * S) / (2 * N) = B * S
#
# The parameter count cancels, so this value depends on neither the model nor
# the hardware. Getting 32 instead of 64 means 4 bytes per parameter (fp32)
# went into the denominator; the GPU model is fp16.
_r = load_results(RESULTS_JSON)
_ai = _r["ai"]
assert abs(_ai - B * S) < 0.5, (
    f"Arithmetic intensity should be batch x seq_len = {B*S} FLOPs/byte, but "
    f"you reported {_ai:g}. If you got {B*S//2}, the weight bytes were counted "
    f"at 4 bytes per parameter (fp32) instead of 2 (fp16).")

# Tie ai back to the byte counts it came from, so the three cannot drift apart.
_implied = (2 * _r["n_params"] * B * S) / _r["weights_bytes"]
_e = rel(_ai, _implied)
assert _e <= TOL, (
    f"ai = {_ai:g} but (2 * n_params * B * S) / weights_bytes = {_implied:g}, "
    f"off by {_e*100:.1f}%.")
print(f"[OK] ai = {_ai:g} FLOPs/byte = batch x seq_len = {B} x {S}, "
      f"consistent with your byte counts")
record("a_ai_equals_64")


---
# Part B — Self-consistency (18 ungraded points)

Each check below recomputes one of your derived values from your own raw
measurements and requires the two to agree within 2%. These relations hold for
any genuine run regardless of hardware.

In [ ]:
# B1 (2 pts): cpu_tps must follow from cpu_ms.
_r = load_results(RESULTS_JSON); _p = _r["perf"]
assert _p["cpu_ms"] > 0, "cpu_ms must be positive."
_exp = B * S / (_p["cpu_ms"] / 1000.0)
_e = rel(_p["cpu_tps"], _exp)
assert _e <= TOL, (
    f"cpu_tps = {_p['cpu_tps']:.1f} but batch*seq/(cpu_ms/1000) = {_exp:.1f}, "
    f"off by {_e*100:.1f}%. Throughput is tokens / elapsed seconds.")
print(f"[OK] cpu_tps consistent with cpu_ms ({_e*100:.2f}% difference)")
record("b_cpu_tps")


In [ ]:
# B2 (2 pts): gpu_tps must follow from gpu_ms.
_r = load_results(RESULTS_JSON); _p = _r["perf"]
assert _p["gpu_ms"] > 0, "gpu_ms must be positive."
_exp = B * S / (_p["gpu_ms"] / 1000.0)
_e = rel(_p["gpu_tps"], _exp)
assert _e <= TOL, (
    f"gpu_tps = {_p['gpu_tps']:.1f} but batch*seq/(gpu_ms/1000) = {_exp:.1f}, "
    f"off by {_e*100:.1f}%.")
print(f"[OK] gpu_tps consistent with gpu_ms ({_e*100:.2f}% difference)")
record("b_gpu_tps")


In [ ]:
# B3 (2 pts): speedup must equal the throughput ratio.
_r = load_results(RESULTS_JSON); _p = _r["perf"]
_exp = _p["gpu_tps"] / _p["cpu_tps"]
_e = rel(_p["speedup"], _exp)
assert _e <= TOL, (
    f"speedup = {_p['speedup']:.2f} but gpu_tps/cpu_tps = {_exp:.2f}, "
    f"off by {_e*100:.1f}%.")
print(f"[OK] speedup consistent with the throughput ratio ({_e*100:.2f}%)")
record("b_speedup_vs_tps")


In [ ]:
# B4 (2 pts): the same speedup must also follow from the two latencies.
_r = load_results(RESULTS_JSON); _p = _r["perf"]
_exp = _p["cpu_ms"] / _p["gpu_ms"]
_e = rel(_p["speedup"], _exp)
assert _e <= TOL, (
    f"speedup = {_p['speedup']:.2f} but cpu_ms/gpu_ms = {_exp:.2f}, off by "
    f"{_e*100:.1f}%. At fixed token count these two ratios must agree.")
print(f"[OK] speedup consistent with the latency ratio ({_e*100:.2f}%)")
record("b_speedup_vs_ms")


In [ ]:
# B5 (3 pts): mem_overhead_ratio must equal peak_bytes / weights_bytes.
# Both byte counts come from your run, so this is a real cross-check rather
# than an identity.
_r = load_results(RESULTS_JSON)
_exp = _r["peak_bytes"] / _r["weights_bytes"]
_e = rel(_r["mem_overhead_ratio"], _exp)
assert _e <= TOL, (
    f"mem_overhead_ratio = {_r['mem_overhead_ratio']:.4f} but peak_bytes / "
    f"weights_bytes = {_exp:.4f}, off by {_e*100:.1f}%.")
print(f"[OK] memory overhead ratio self-consistent "
      f"({_r['mem_overhead_ratio']:.2f}x)")
record("b_mem_ratio")


In [ ]:
# B6 (2 pts): landscape_row["forward_latency_ms"] mirrors perf["gpu_ms"].
_r = load_results(RESULTS_JSON)
_e = rel(num(_r["landscape_row"]["forward_latency_ms"]), _r["perf"]["gpu_ms"])
assert _e <= TOL, (
    f"landscape_row['forward_latency_ms'] should be your measured GPU latency "
    f"perf['gpu_ms'] = {_r['perf']['gpu_ms']:.2f} ms (off by {_e*100:.1f}%).")
print("[OK] landscape latency matches the GPU measurement")
record("b_ls_latency")


In [ ]:
# B7 (1 pt): landscape_row["throughput_tps"] mirrors perf["gpu_tps"].
_r = load_results(RESULTS_JSON)
_e = rel(num(_r["landscape_row"]["throughput_tps"]), _r["perf"]["gpu_tps"])
assert _e <= TOL, (
    f"landscape_row['throughput_tps'] should be perf['gpu_tps'] = "
    f"{_r['perf']['gpu_tps']:.1f} (off by {_e*100:.1f}%).")
print("[OK] landscape throughput matches the GPU measurement")
record("b_ls_tps")


In [ ]:
# B8 (2 pts): unit check. peak_memory_MB is megabytes; peak_bytes is bytes.
_r = load_results(RESULTS_JSON)
_exp = _r["peak_bytes"] / 1e6
_e = rel(num(_r["landscape_row"]["peak_memory_MB"]), _exp)
assert _e <= TOL, (
    f"landscape_row['peak_memory_MB'] = "
    f"{num(_r['landscape_row']['peak_memory_MB']):g} but peak_bytes/1e6 = "
    f"{_exp:.1f}. Unit mix-up: peak_bytes is in BYTES, peak_memory_MB in "
    f"MEGABYTES.")
print(f"[OK] peak memory units consistent ({_exp:.1f} MB)")
record("b_ls_peak_mb")


In [ ]:
# B9 (2 pts): dominant_op must be the top entry of your own top3 list.
_r = load_results(RESULTS_JSON)
_dom = str(_r["landscape_row"]["dominant_op"]).strip()
_top = _r["top3"][0][0].strip()
assert _dom == _top, (
    f"landscape_row['dominant_op'] is '{_dom}' but the first entry of top3 is "
    f"'{_top}'. The dominant operator is top3[0][0].")
if not is_gemm(_dom):
    print(f"[note] '{_dom}' is not a matrix-multiply kernel. That is unusual "
          f"but can happen on some torch builds; no points lost.")
print(f"[OK] dominant_op matches top3[0][0]: {_dom}")
record("b_ls_dominant_op")


---
# Part C — Physics and plausibility (8 ungraded points)

Wide sanity bands. These confirm you actually ran on a GPU runtime and that the
ordering of your measurements makes physical sense. The bands are deliberately
generous to accommodate whatever hardware Colab assigned you and however
contended the VM was.

In [ ]:
# C1 (1 pt): the CPU forward pass should be slower than the GPU one.
_r = load_results(RESULTS_JSON); _p = _r["perf"]
assert _p["cpu_ms"] > _p["gpu_ms"], (
    f"cpu_ms ({_p['cpu_ms']:.1f}) should exceed gpu_ms ({_p['gpu_ms']:.1f}). "
    f"Did you benchmark model_cpu with inputs_cpu and model_gpu with inputs_gpu?")
print(f"[OK] CPU slower than GPU, as expected")
record("c_cpu_gt_gpu")


In [ ]:
# C2 (1 pt): CPU latency in a plausible band (Colab CPU VMs vary a lot).
_v = load_results(RESULTS_JSON)["perf"]["cpu_ms"]
assert 50 < _v < 60000, (
    f"cpu_ms = {_v:.1f} ms is outside the plausible range 50-60000 ms for a "
    f"0.5B model forward pass on a Colab CPU.")
print(f"[OK] cpu_ms = {_v:.1f} ms is plausible")
record("c_cpu_ms_plausible")


In [ ]:
# C3 (1 pt): GPU latency in a plausible band (A100 ~8 ms, throttled T4 slower).
_v = load_results(RESULTS_JSON)["perf"]["gpu_ms"]
assert 0.5 < _v < 2000, (
    f"gpu_ms = {_v:.1f} ms is outside the plausible range 0.5-2000 ms. If this "
    f"is very large, check that the model was really on the GPU.")
print(f"[OK] gpu_ms = {_v:.1f} ms is plausible")
record("c_gpu_ms_plausible")


In [ ]:
# C4 (2 pts): confirm the lab really ran on a GPU runtime.
# This is the gate for the whole comparison: with no GPU there is no CPU-vs-GPU
# result to report, so it carries more weight than the other band checks.
_r = load_results(RESULTS_JSON); _p = _r["perf"]
_v = _p["speedup"]
assert _v > 2.0, (
    f"speedup = {_v:.2f}x. A GPU runtime gives a large margin on this workload "
    f"(tens of times on a T4). A value near 1x means the lab ran on CPU only: "
    f"set Runtime -> Change runtime type -> T4 GPU in Colab and re-run it.")
# fp16 on a GPU should also be clearly faster in absolute terms than fp32 on a
# Colab CPU; a "GPU" latency in CPU territory means the model stayed on the host.
assert _p["gpu_ms"] < 500, (
    f"gpu_ms = {_p['gpu_ms']:.1f} ms is in CPU territory for this model. Check "
    f"that model_gpu really was moved to CUDA (.cuda()).")
print(f"[OK] GPU runtime confirmed: speedup = {_v:.1f}x, gpu_ms = {_p['gpu_ms']:.1f} ms")
record("c_speedup_gt")


In [ ]:
# C5 (1 pt): peak memory must exceed the weights alone.
_r = load_results(RESULTS_JSON)
assert _r["mem_overhead_ratio"] >= 1.0, (
    f"mem_overhead_ratio = {_r['mem_overhead_ratio']:.2f} is below 1.0, which "
    f"would mean peak memory was less than the weights themselves.")
assert _r["peak_bytes"] > _r["weights_bytes"], (
    "peak_bytes should exceed the model weight bytes: activations, attention "
    "intermediates and logits all add to the peak.")
print(f"[OK] peak exceeds weights ({_r['mem_overhead_ratio']:.2f}x)")
record("c_ratio_ge_1")


In [ ]:
# C6 (2 pts): the roofline verdict.
#
# ai = 64 sits below the ridge point of every GPU Colab hands out, so
# "memory-bound" is hardware-independent. What is NOT hardware-independent is
# the ridge point itself, so this also checks that the ridge you reported
# belongs to the GPU your run actually used.
_r = load_results(RESULTS_JSON)
_rp, _ai = _r["ridge_point"], _r["ai"]

_lo, _hi = RIDGE_BAND
assert _lo <= _rp <= _hi, (
    f"ridge_point = {_rp:.0f} is outside the range for Colab GPUs "
    f"({_lo:.0f}-{_hi:.0f}). Compute it as "
    f"peak_fp16_tflops / peak_bandwidth_tb_s.")

# If we can identify the GPU, the ridge must match that family's value.
_fam = gpu_family(_r["gpu_name"]) | gpu_family(_r["landscape_row"]["accelerator"])
_known = _fam & set(RIDGE_TABLE)
if len(_known) == 1:
    _k = _known.pop()
    _e = rel(_rp, RIDGE_TABLE[_k])
    assert _e <= 0.05, (
        f"Your run reports a {_k.upper()}, whose ridge point is "
        f"{RIDGE_TABLE[_k]:.0f} FLOPs/byte, but you gave ridge_point = "
        f"{_rp:.0f} (off by {_e*100:.0f}%). Use the row for the GPU you "
        f"actually ran on.")
    print(f"[OK] ridge_point {_rp:.0f} matches the {_k.upper()} your run reported")
else:
    print(f"[note] could not identify a known GPU in '{_r['gpu_name']}'; "
          f"accepting any ridge in {_lo:.0f}-{_hi:.0f}. No points lost.")

assert _ai < _rp, (
    f"With ai = {_ai:g} below the ridge point {_rp:.0f}, this workload is "
    f"memory-bound.")
assert _r["verdict"] == "MEMORY-BOUND", (
    f"verdict should be 'MEMORY-BOUND' (ai = {_ai:g} < ridge = {_rp:.0f}), "
    f"got {_r['verdict']!r}.")
print(f"[OK] memory-bound confirmed: ai {_ai:g} < ridge {_rp:.0f}")
record("c_memory_bound")


In [ ]:
# === Your self-assessment score (read-only readout -- nothing here is graded) ===
# Run this after running every cell above (Kernel -> Restart & Run All).
# It summarizes the checks above so you can see which parts of the lab went
# well. These points are ungraded and do not affect your course grade.
try:
    _rows, _earned, _possible = [], {}, {}
    for _gid, _part, _pts in MANIFEST:
        _ok = _gid in _SCORE
        _earned[_part]   = _earned.get(_part, 0)   + (_pts if _ok else 0)
        _possible[_part] = _possible.get(_part, 0) + _pts
        _rows.append((_part, _gid, _pts, _ok))

    _parts  = [p for p in ("A", "B", "C") if p in _possible]
    _auto_e = sum(_earned[p]   for p in _parts)
    _auto_p = sum(_possible[p] for p in _parts)

    print("=" * 68)
    print("  YOUR SELF-ASSESSMENT SCORE  (40 ungraded points)")
    print("=" * 68)
    for _p in _parts:
        _bar = "#" * int(round(18 * _earned[_p] / max(_possible[_p], 1)))
        print(f"  {_p} - {PART_NAMES[_p]:26s} {_earned[_p]:3d} / {_possible[_p]:<3d} {_bar}")
    print(f"  {'-' * 64}")
    print(f"  TOTAL                          {_auto_e:3d} / {_auto_p:<3d}")

    # Per-check detail: every graded cell, what it checks, and the points it
    # earned. This is the full breakdown behind the part totals above.
    print("=" * 68)
    print("  DETAIL - every check, in order")
    print("=" * 68)
    _n = {}
    for _p, _g, _pt, _ok in _rows:
        _n[_p] = _n.get(_p, 0) + 1
        _tag   = f"{_p}{_n[_p]}"
        _mark  = "PASS" if _ok else "FAIL"
        _got   = _pt if _ok else 0
        _what  = CHECK_LABELS.get(_g, _g)
        print(f"  {_tag:<4s} {_mark}  {_got}/{_pt}  {_what}")
    print(f"  {'-' * 64}")
    print(f"  TOTAL      {_auto_e:2d}/{_auto_p:<2d}")

    _failed = [(p, g) for p, g, _, ok in _rows if not ok]
    if _failed:
        print("-" * 68)
        print("  Cells not yet passing - scroll up to their error messages:")
        for _p, _g in _failed:
            print(f"    [{_p}] {_g}  -  {CHECK_LABELS.get(_g, '')}")
        print("  Each is independent: fixing one cannot cost you another.")
    else:
        print("-" * 68)
        print("  Every autograded cell passed.")

    print("=" * 68)
    print("  Wrapping up")
    print("  1. Kernel -> Restart & Run All, then re-read this table. If you")
    print("     ran cells out of order, the numbers above can be stale.")
    print("  2. These 40 points are UNGRADED self-assessment feedback, auto-")
    print("     checked from your pasted JSON. No write-up, no human review,")
    print("     and nothing here affects your course grade.")
    print("  3. The GRADED item is the Concept Check quiz: a separate auto-")
    print("     graded Coursera Assignment worth 10 points. It is mandatory")
    print("     and needs 80% to pass. Do not skip it.")
    print("=" * 68)
except Exception as _e:
    # A courtesy readout must never be the reason a passing student panics.
    print(f"[score readout unavailable: {type(_e).__name__}: {_e}]")
    print("Your check cells above are unaffected - each prints [OK] when it passes.")


---
## Wrapping up

1. **Run every cell top to bottom** (Kernel → Restart & Run All). Each check
   prints `[OK]` when it passes, and the **score table** just above reports your
   40-point self-assessment total.
2. Any cell that raises an error scores 0 **for that cell only** — the checks are
   independent, so fix what you can; the rest of your self-assessment still counts.
3. Confirm the echo table under your pasted JSON matches your Colab output.
4. **Take the Concept Check quiz.** It is a separate auto-graded Coursera
   **Graded Assignment** worth **10 graded points**, it is mandatory, and you
   need **80%** to pass. The 40 points in this notebook are ungraded feedback;
   the quiz is what counts toward your grade.

### Troubleshooting

| Symptom | Fix |
|---|---|
| `Could not parse your results` | Paste as plain text; curly "smart quotes" break JSON. Copy from `{` through `}`. |
| `Your pasted JSON looks incomplete` | You copied only part of the file. Select all of it. |
| `Key(s) [...] are missing` naming `ai`/`n_params`/`verdict` | Your Colab notebook is an older version of Lab 1. Re-download the student notebook, re-run it, and copy the new JSON. |
| `peak_bytes looks like megabytes` | Use the raw `torch.cuda.max_memory_allocated()` value, not the MB figure. |
| `exported n_params ... but peak_bytes / ratio / 2` | One of the four memory fields was edited or mis-scaled. Re-run Parts 4 and 7 in Colab rather than adjusting values by hand. |
| `speedup = 1.0x` or CPU faster | Colab ran without a GPU. Set **Runtime → Change runtime type → T4 GPU** and re-run the lab. |
| `'llm_forward'` still in top3 | Filter that label out before sorting for the top 3. |
| Arithmetic intensity is 32, not 64 | You used 4 bytes/param (fp32). The GPU model is fp16: 2 bytes. |
| `ridge_point ... does not match the GPU` | Use the table row for the GPU your run reported, not the T4 row by default. |

Your Colab notebook stays yours — nothing from it is uploaded except the values
you pasted here.

---

*Course: Dataflow Architectures for AI, Week 1 Lab, v1.0 · © 2026 Dr. Kaoutar El Maghraoui*
